# Figure 3: QQNM frequency and damping-time contours

Input: `data/fig03_spectral_freqtau_grids.npz`. The released grids are the exact 50-by-50 Gaussian-smoothed densities computed from all posterior samples, using the 0.5--99.5 percentile range plus 25% padding. Frequencies are in Hz and damping times in ms.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', 'Wswiglal-redir-stdio')

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data').is_dir():
    raise RuntimeError('Run this notebook from the repository root.')

fig_width = 3 * 246.0 / 72.27
fig_height = fig_width * (np.sqrt(5) - 1.0) / 2.0
mpl.rcParams.update({
    'axes.labelsize': 24,
    'font.family': 'serif',
    'font.serif': 'Computer Modern Roman',
    'font.size': 24,
    'legend.fontsize': 20,
    'xtick.labelsize': 24,
    'ytick.labelsize': 24,
    'axes.grid': True,
    'text.usetex': True,
    'savefig.dpi': 100,
    'lines.markersize': 14,
    'figure.figsize': [fig_width, fig_height],
})

data = np.load(ROOT / 'data/fig03_spectral_freqtau_grids.npz')
qqnm_modes = ['220220', '220221', '221221']
kerr_modes = ['220', '221', '440', '441']
colors = plt.cm.tab10(np.arange(10))
line_styles = {4: 'solid', 5: 'dashed'}
kerr_styles = {
    '220': dict(color='k', mfc='k'),
    '221': dict(color='k', mfc='none'),
    '440': dict(color='gray', mfc='gray'),
    '441': dict(color='gray', mfc='none'),
}

In [ ]:
fig, ax = plt.subplots()
qqnm_handles = []
for index, mode in enumerate(qqnm_modes):
    for start, line_style in line_styles.items():
        prefix = f'{start}mf_{mode}'
        ax.contour(
            data[f'x_{prefix}'], data[f'y_{prefix}'], data[f'density_{prefix}'],
            levels=[float(data[f'level90_{prefix}'])], colors=[colors[index]],
            linewidths=1.8, linestyles=line_style)
    qqnm_handles.append(plt.Line2D(
        [0], [0], color=colors[index], lw=1.8,
        label=f'${mode[:3]}\\times{mode[3:]}$'))

for mode in kerr_modes:
    frequency, damping_time = data[f'kerr_{mode}']
    style = kerr_styles[mode]
    ax.plot(frequency, damping_time, marker='*', ms=22, ls='none',
            color=style['color'], mfc=style['mfc'], mew=2.0, zorder=6)

kerr_handles = [plt.Line2D(
    [0], [0], marker='*', ms=16, ls='none', color=kerr_styles[mode]['color'],
    mfc=kerr_styles[mode]['mfc'], mew=2.0, label=f'{mode} (Kerr)')
    for mode in kerr_modes]
start_handles = [plt.Line2D(
    [0], [0], color='k', lw=1.8, ls=line_style,
    label=f'{start} $M_\\mathrm{{f}}$')
    for start, line_style in line_styles.items()]

legend_qqnm = ax.legend(handles=qqnm_handles, loc='upper left', fontsize=24, framealpha=0.9)
ax.add_artist(legend_qqnm)
legend_kerr = ax.legend(handles=kerr_handles, loc='upper right', fontsize=24, framealpha=0.9)
ax.add_artist(legend_kerr)
ax.legend(handles=start_handles, loc='lower right', fontsize=18, framealpha=0.9)
ax.set_xlabel('Frequency / Hz')
ax.set_ylabel('Damping time / ms')
ax.set_ylim(0, 7.5)

output = ROOT / 'figures/spectral_freqtau_contour_wls10modes_gaussian_4Mf.pdf'
fig.savefig(output, bbox_inches='tight')
plt.show()